# 程序性记忆（Procedural Memory）

> 捕捉学到的过程、工作流程和「怎么做」的知识，让智能体随经验变强。

回想在新厨房第一次煮咖啡：你打开每个橱柜，翻遍每个抽屉，跟咖啡机较劲。到第十个早晨，你的双手已经自动运转，整个流程内化成了程序。

大多数记忆技术存储*发生了什么*（情景记忆）或*什么为真*（语义记忆）。程序性记忆存储的是*怎么做事*。它捕捉智能体在经验中发现的分步策略和工作流（能达成目标的一串动作）。

问题在这里。智能体靠推理和工具调用解出一道复杂任务时，那条成功的动作序列就是宝贵的知识。没有程序性记忆，智能体每次都要从头推导解法。有了程序性记忆，它检索学到的过程，再适配到新场景。

Voyager（Wang et al., 2023）这类系统已经证明了这一点。那个 Minecraft 智能体会编写可复用的代码化技能，存进一个技能库（skill library，有条理的过程集合）。遇到类似任务时，它直接取回存好的技能，而不是从零开始。

**在本 notebook 中，你将从零构建一个程序性记忆系统。** 它会：
1. 从成功的执行轨迹（execution trace，智能体动作的日志）中抽取可复用的过程。
2. 把过程存进按任务类型索引的技能库。
3. 新任务到来时，检索匹配的过程。
4. 把检索到的过程适配到新情境。


## 关键概念

- **执行轨迹（execution trace）**：智能体在一次任务中所有动作的记录，包括执行的动作、调用的工具、给出的输入和收到的输出。
- **过程抽取（procedure extraction）**：从一条具体的执行轨迹中识别出可复用的动作序列。LLM（large language model，处理文本的 AI）审阅轨迹，把它蒸馏成一份干净的模板。
- **工作流模板（workflow template）**：参数化的过程。具体值（文件名、服务器地址、应用名）替换成 `{app_name}` 这样的占位符，工作流就能套用到新输入上，不用重写。
- **技能库（skill library）**：按任务类型索引、有条理的过程集合。智能体解决的新任务类型越多，库就越大。
- **过程适配（procedure adaptation）**：把模板里的占位符填上新任务的具体值。适配后的过程可以直接执行。
- **过程泛化（procedure generalization）**：把具体过程抽象成更通用的形式。「部署 weather-api 到 deploy.cloudhost.io」变成「部署 {app_name} 到 {server_host}」。


## Architecture

<p align="center">
  <img src="../../images/diagrams/11_procedural_memory.svg" alt="程序性记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    Task["Task Execution\n(step-by-step)"] --> SuccessCheck{"Task\nSucceeded?"}
    SuccessCheck -- No --> Discard["Discard / Log\nfor debugging"]
    SuccessCheck -- Yes --> Extractor["Procedure Extractor\n(LLM call)"]
    Extractor --> Template["Workflow Template\n(parameterized steps)"]
    Template --> Library["Skill Library\n(indexed by task type)"]
    NewTask["New Task"] --> Retrieval["Retrieval\n(task-type matching)"]
    Library --> Retrieval
    Retrieval --> Adaptation["Adaptation\n(fill parameters)"]
    Adaptation --> Execution["Execute Adapted\nProcedure"]
```

</details>

**数据流：** 智能体完成一个任务后，执行轨迹送入过程抽取器（一次 LLM 调用）。抽取器把动作序列蒸馏成参数化的工作流模板。模板存进技能库，按任务类型和元数据索引。

新任务到来时，检索引擎拿它去匹配库里的过程。找到相关的过程，就把占位符填成具体值来适配模板。没有匹配，智能体就从零解决任务。然后循环往复。

## 环境准备

LLM 调用需要 OpenAI Python SDK，加载 `.env` 文件里的 API key 需要 `python-dotenv`。


In [ ]:
%pip install -q python-dotenv deepagents langchain-openai

加载 API key 并创建 OpenAI 客户端。运行前，先在你的 `.env` 文件里设置 `LLM_API_KEY`。


In [ ]:
import os
import json
from dataclasses import dataclass, field
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们将构建三个组件：

1. **过程数据模型。** 一个 dataclass，保存过程的名称、步骤、参数和元数据。
2. **LLM 提示词模板。** 三条提示词，分别驱动抽取、检索和适配。
3. **`ProceduralMemory` 类。** 把一切串起来的核心引擎。


### 过程数据模型

把它想象成食谱盒里的一张卡片。每张卡片记录菜名、类别（前菜、主菜、甜点）、需要的食材，以及编号的步骤。我们的 `Procedure` dataclass 就是这张卡片。

`steps` 字段存放参数化的动作描述。`parameters` 字段列出这些步骤里用到的占位符名称。`preconditions` 字段记录执行前必须成立的条件。


In [ ]:
@dataclass
class Procedure:
    """从成功任务中抽取的、可复用的参数化过程。"""

    name: str
    task_type: str
    description: str
    steps: list[str]           # 参数化的动作描述
    parameters: list[str]      # 占位符名称（如 ["app_name", "server_host"]）
    preconditions: list[str]   # 执行前必须成立的条件
    success_count: int = 0
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def display(self) -> None:
        """打印这份过程的可读摘要。"""
        print(f"Procedure: {self.name}")
        print(f"  Type: {self.task_type}")
        print(f"  Description: {self.description}")
        print(f"  Parameters: {self.parameters}")
        print(f"  Preconditions: {self.preconditions}")
        print(f"  Steps:")
        for i, step in enumerate(self.steps, 1):
            print(f"    {i}. {step}")
        print(f"  Success count: {self.success_count}")

### LLM 提示词模板

系统做三次不同的 LLM 调用，各自配一条提示词。

1. **抽取提示词**：读入执行轨迹，产出一个参数化的过程。
2. **检索提示词**：把新任务和技能库比对，挑出最匹配的过程。
3. **适配提示词**：把过程的占位符填上新任务情境里的值。

三条提示词都要求 JSON 输出，解析起来更可靠。


In [ ]:
EXTRACTION_PROMPT = """\
You are a procedure extractor. You receive an execution trace from a \
successful task. Your job: distill it into a reusable, parameterized procedure.

Rules:
- Replace specific values (file names, URLs, server addresses, app names) \
with descriptive placeholders like {app_name}, {server_host}, {file_path}.
- Keep the steps in execution order.
- Remove noise (debug output, retries, irrelevant side-effects).
- Identify preconditions (what must be true before this procedure can run).

Return a JSON object with these fields:
- name: short name for the procedure (e.g., "Deploy containerized app")
- task_type: category (e.g., "deployment", "data_pipeline", "api_integration")
- description: one-sentence summary of what this procedure accomplishes
- steps: list of parameterized step descriptions (strings)
- parameters: list of placeholder names used in the steps
- preconditions: list of conditions that must hold before execution
"""

RETRIEVAL_PROMPT = """\
You are a procedure retrieval engine. You receive a task description and a \
skill library (a list of stored procedures with index numbers).

Pick the best matching procedure for the task.

Return a JSON object with:
- best_match_index: integer index of the best match, or null if none is relevant
- confidence: float between 0.0 and 1.0
- reasoning: one sentence explaining your choice
"""

ADAPTATION_PROMPT = """\
You are a procedure adaptation engine. You receive a parameterized procedure \
and a new task context. Fill in the placeholders with concrete values from \
the context.

Return a JSON object with:
- adapted_steps: list of concrete step descriptions (placeholders filled in)
- parameter_values: object mapping each parameter name to its concrete value
"""

### ProceduralMemory 类

这个类把抽取、存储、检索和适配串在一起。技能库就是一个 Python 列表。生产系统里，你会用带向量搜索（比较数值表示来查找相似项）的数据库来加速检索。出于学习目的，列表加基于 LLM 的匹配就够用了。


In [ ]:
class ProceduralMemory:
    """一个技能库，从成功的任务执行中学习过程。"""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
        self.skill_library: list[Procedure] = []

    # ── 抽取 ───────────────────────────────────────────────
    def extract_procedure(self, execution_trace: str) -> Procedure:
        """用 LLM 从执行轨迹中抽取一个可复用的过程。"""
        response = llm.invoke([
                {"role": "system", "content": EXTRACTION_PROMPT},
                {"role": "user", "content": f"Execution trace:\n{execution_trace}"},
            ])
        data = json.loads(to_text(response.content))
        return Procedure(
            name=data["name"],
            task_type=data["task_type"],
            description=data["description"],
            steps=data["steps"],
            parameters=data["parameters"],
            preconditions=data.get("preconditions", []),
        )

    # ── 存储 ──────────────────────────────────────────────────
    def store_procedure(self, procedure: Procedure) -> str:
        """把过程加入技能库。若已有相似过程则合并。"""
        existing = self._find_similar(procedure)
        if existing:
            existing.success_count += 1
            return f"Merged with existing: '{existing.name}' (count: {existing.success_count})"
        self.skill_library.append(procedure)
        return f"Stored new procedure: '{procedure.name}'"

接下来添加检索和适配方法。检索用 LLM 把新任务描述和所有已存的过程逐一比对。适配把过程的占位符填上新任务情境里的具体值。


In [ ]:
# ── 检索 ────────────────────────────────────────────────
def _retrieve_procedure(self, task_description: str) -> tuple[Procedure | None, dict]:
    """为新任务找到最匹配的过程。"""
    if not self.skill_library:
        return None, {"reasoning": "Skill library is empty."}

    library_summary = "\n".join(
        f"[{i}] {p.name} (type: {p.task_type}): {p.description}"
        for i, p in enumerate(self.skill_library)
    )

    response = llm.invoke([
            {"role": "system", "content": RETRIEVAL_PROMPT},
            {"role": "user", "content": (
                f"Task: {task_description}\n\n"
                f"Skill library:\n{library_summary}"
            )},
        ])
    result = json.loads(to_text(response.content))
    idx = result.get("best_match_index")
    if idx is not None and 0 <= idx < len(self.skill_library):
        return self.skill_library[idx], result
    return None, result

# ── 适配 ───────────────────────────────────────────────
def _adapt_procedure(self, procedure: Procedure, task_context: str) -> dict:
    """通过填充占位符，把过程的步骤适配到新情境。"""
    steps_text = "\n".join(
        f"  {i+1}. {s}" for i, s in enumerate(procedure.steps)
    )

    response = llm.invoke([
            {"role": "system", "content": ADAPTATION_PROMPT},
            {"role": "user", "content": (
                f"Procedure: {procedure.name}\n"
                f"Steps:\n{steps_text}\n\n"
                f"Parameters: {procedure.parameters}\n\n"
                f"New task context: {task_context}"
            )},
        ])
    return json.loads(to_text(response.content))

ProceduralMemory.retrieve_procedure = _retrieve_procedure

ProceduralMemory.adapt_procedure = _adapt_procedure


`learn_from_execution` 方法把整个循环串起来：任务成功时，一次调用就完成过程的抽取和存储。辅助方法负责重复检测和列出库内容。


In [ ]:
# ── 完整学习循环 ───────────────────────────────────────
def _learn_from_execution(
    self, execution_trace: str, success: bool
) -> Procedure | None:
    """任务成功时，抽取并存储一个过程。"""
    if not success:
        print("Task failed. Skipping procedure extraction.")
        return None
    procedure = self.extract_procedure(execution_trace)
    status = self.store_procedure(procedure)
    print(status)
    return procedure

# ── 内部辅助方法 ─────────────────────────────────────────
def __find_similar(self, procedure: Procedure) -> Procedure | None:
    """检查是否已存在相似的过程（按类型和名称判断）。"""
    for p in self.skill_library:
        if (
            p.task_type == procedure.task_type
            and p.name.lower() == procedure.name.lower()
        ):
            return p
    return None

def _list_procedures(self) -> None:
    """打印技能库里的所有过程。"""
    if not self.skill_library:
        print("Skill library is empty.")
        return
    for i, p in enumerate(self.skill_library):
        print(f"\n{'='*60}")
        print(f"[{i}] ", end="")
        p.display()

ProceduralMemory.learn_from_execution = _learn_from_execution

ProceduralMemory._find_similar = __find_similar

ProceduralMemory.list_procedures = _list_procedures


## 运行示例

来看程序性记忆的实战，分三个场景：

1. 智能体部署一个 web API。我们抽取并存储部署过程。
2. 一个类似的部署任务到来。我们检索存储的过程并适配它。
3. 一个数据管道任务运行。我们抽取第二个过程，库随之变大。


### 场景 1：学习一个部署过程

下面是一条执行轨迹，来自一个把 Flask API 部署到云服务器的智能体。每一行记录智能体执行的一个动作。


In [ ]:
deployment_trace = """\
1. Created project directory "weather-api"
2. Initialized Python 3.11 virtual environment in weather-api/venv
3. Installed dependencies: flask, requests, gunicorn (pip install)
4. Wrote application code in weather-api/app.py with /forecast and /health endpoints
5. Wrote unit tests in weather-api/test_app.py
6. Ran pytest on test_app.py - all 8 tests passed
7. Created Dockerfile with python:3.11-slim base image
8. Built Docker image tagged "weather-api:v1"
9. Pushed image to container registry at registry.cloudhost.io/weather-api:v1
10. Connected via SSH to server at deploy.cloudhost.io
11. Pulled the Docker image on the server
12. Started container with port mapping 80:5000 using docker-compose
13. Verified health check at https://deploy.cloudhost.io/health returned 200 OK
"""

print(deployment_trace)

我们把轨迹交给 `learn_from_execution`。LLM 抽取出一个参数化的过程，系统把它存进技能库。


In [ ]:
memory = ProceduralMemory(client=client, model=MODEL)

procedure = memory.learn_from_execution(deployment_trace, success=True)
print()
procedure.display()

### 场景 2：为新任务检索并适配

新任务来了：把另一个 API 部署到另一台服务器。智能体不从零开始，而是先查自己的技能库。


In [ ]:
new_task = "Deploy the inventory-service (a FastAPI app) to production server prod.mycompany.com"

matched_procedure, retrieval_info = memory.retrieve_procedure(new_task)

print("Retrieval result:")
print(f"  Matched: {matched_procedure.name if matched_procedure else 'None'}")
print(f"  Confidence: {retrieval_info.get('confidence', 'N/A')}")
print(f"  Reasoning: {retrieval_info.get('reasoning', 'N/A')}")

检索引擎找到了我们的部署过程。现在适配它。适配步骤把每个占位符填上新任务里的具体值。


In [ ]:
task_context = (
    "App name: inventory-service. "
    "Framework: FastAPI. "
    "Server: prod.mycompany.com. "
    "Registry: docker.mycompany.com. "
    "Port mapping: 80:8000. "
    "Python version: 3.12."
)

adaptation = memory.adapt_procedure(matched_procedure, task_context)

print("Adapted steps for the new task:\n")
for i, step in enumerate(adaptation["adapted_steps"], 1):
    print(f"  {i}. {step}")

print(f"\nParameter values:")
print(json.dumps(adaptation.get("parameter_values", {}), indent=2))

### 场景 3：扩充技能库

现在智能体运行一个数据管道任务。这是另一种任务类型，所以系统会在部署过程旁边新建一个过程。


In [ ]:
pipeline_trace = """\
1. Connected to PostgreSQL database at db.analytics.io:5432
2. Queried the 'sales' table for records where date >= '2024-01-01' and date < '2024-02-01'
3. Loaded 15,420 rows into a pandas DataFrame
4. Cleaned data: dropped 23 rows with null revenue, converted currency columns to USD
5. Grouped by region and product_category, computed sum of revenue and count of orders
6. Generated a summary CSV report with 48 rows (one per region-category pair)
7. Uploaded report to S3 at s3://company-reports/monthly/2024-01-sales-summary.csv
8. Sent notification email to analytics-team@company.com with the S3 link
"""

pipeline_procedure = memory.learn_from_execution(pipeline_trace, success=True)

技能库里现在有两个来自不同领域的过程。我们把它们都看一遍。


In [ ]:
memory.list_procedures()

### 选择性检索

库里有多个过程之后，检索引擎必须挑对那一个。我们用每个领域的任务分别测一测。


In [ ]:
# 部署类任务应匹配部署过程
task_a = "Deploy the user-auth microservice to staging server staging.myapp.io"
match_a, info_a = memory.retrieve_procedure(task_a)
print(f"Task: {task_a}")
print(f"  Matched: {match_a.name if match_a else 'None'}")
print(f"  Reasoning: {info_a.get('reasoning', 'N/A')}")

print()

# 数据类任务应匹配数据管道过程
task_b = "Generate the February 2024 sales report from the analytics database"
match_b, info_b = memory.retrieve_procedure(task_b)
print(f"Task: {task_b}")
print(f"  Matched: {match_b.name if match_b else 'None'}")
print(f"  Reasoning: {info_b.get('reasoning', 'N/A')}")

## 权衡

### 程序性记忆表现出色的场景

- **重复的多步任务。** 部署服务、跑数据管道、集成 API 的智能体，会反复执行相似的工作流。存下过程，就省去每次重新推导步骤。
- **跨会话的技能迁移。** 一个会话里抽出的过程，下个会话仍然可用。不用重训或微调，智能体的能力随时间增长。
- **执行更快。** 检索存储的过程再适配，比从零开始多步推理花的 LLM 调用更少。

### 它会失效的场景

- **全新任务。** 如果每个任务都是独一无二的，技能库永远用不上，抽取成本变成纯开销。
- **脆弱的过程。** 从一次成功运行抽出的过程未必能泛化。环境一变（不同的操作系统、不同的 API 版本），存好的步骤可能毫无征兆地失败。你需要反馈循环和版本管理来应对。
- **检索出错。** 基于 LLM 的检索可能匹配到错误的过程，库变大后尤其如此。生产系统需要置信度阈值和回退行为（置信度低就从零解决）。
- **过时的过程。** 工具和 API 在变，过程会过时。没有失效或更新机制，库里会积累不可靠的知识。


## 延伸阅读

- [Wang et al., "Voyager: An Open-Ended Embodied Agent with Large Language Models," 2023](https://arxiv.org/abs/2305.16291) - 一个 Minecraft 智能体，构建了由代码化过程组成的可复用技能库。是与本 notebook 实现最接近的现有系统。
- [Sumers et al., "Cognitive Architectures for Language Agents," 2023](https://arxiv.org/abs/2309.02427) - LLM 智能体认知架构的综述。把程序性记忆作为核心组件，与情景记忆、语义记忆一起介绍。
- [Anderson, "Acquisition of Cognitive Skill," *Psychological Review*, 1982](https://doi.org/10.1037/0033-295X.89.4.369) - 心理学奠基论文，讲人如何从陈述性知识（「知道是什么」）过渡到程序性知识（「知道怎么做」）。
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," 2023](https://arxiv.org/abs/2304.03442) - 通过记忆和反思发展出行为模式的模拟智能体。一种涌现的程序性知识。


---

*← 上一篇：[10 - 语义记忆](../10_semantic_memory/) · 下一篇：[12 - 工作记忆与上下文窗口](../12_working_memory_context_window/) →*


## 🧪 动手试试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：过程版本管理
给 `Procedure` dataclass 加一个 `version` 字段和一个 `history` 列表。每次 `store_procedure()` 更新已有过程时，递增版本号并归档旧版本。跑过几轮学习后，打印版本历史。

### 挑战 2：适配成功率
创建 5 个任务变体，都需要适配同一个基础过程。对每个变体运行 `adapt_procedure()`，然后执行适配后的过程并评定成败。报告适配成功率，并找出哪类适配最常失败。

### 挑战 3：记忆管理过程
用 `ProceduralMemory` 存储关于何时摘要、何时遗忘、何时巩固记忆的过程。把这些过程喂回一个管理自身记忆的智能体。这是程序性记忆与 17 记忆路由的路由逻辑的结合。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--11-procedural-memory--procedural-memory)